# AgentCore Runtime에서 대용량 Multi-Modal Payload 처리

## 개요

이 튜토리얼에서는 Amazon Bedrock AgentCore Runtime이 Excel 파일과 image 같은 multi-modal 콘텐츠를 포함하여 최대 100MB의 대용량 payload를 처리하는 방법을 보여줍니다. AgentCore Runtime은 rich media 콘텐츠와 대규모 dataset을 원활하게 처리하도록 설계되었습니다.

### 튜토리얼 세부 정보

|항목| 세부 정보|
|:--------------------|:---------------------------------------------------------------------------------|
| 튜토리얼 유형       | 대용량 Payload 및 Multi-Modal 처리|
| 에이전트 유형       | 단일           |
| Agentic Framework   | Strands Agents |
| LLM 모델            | Anthropic Claude Haiku 4.5 |
| 튜토리얼 구성 요소  | 대용량 파일 처리, Image 분석, Excel 데이터 처리 |
| 튜토리얼 분야       | 데이터 분석 및 Multi-Modal AI                                                   |
| 예제 난이도         | 중간                                                                             |
| 사용 SDK            | Amazon BedrockAgentCore Python SDK|

### 주요 기능

* **대용량 Payload 지원**: 최대 100MB 크기의 파일 처리
* **Multi-Modal 처리**: Excel 파일, image, text 동시 처리
* **데이터 분석**: structured data와 시각적 콘텐츠에서 insight 추출
* **Base64 Encoding**: JSON payload를 통한 binary data의 안전한 전송

## 사전 요구 사항

* Python 3.10+
* 구성된 AWS credentials
* 실행 중인 Docker
* 테스트용 sample Excel 파일 및 image

In [ ]:
!pip install --force-reinstall -U -r requirements.txt --quiet

## Sample Data 파일 생성

대용량 payload 처리를 보여주기 위한 sample Excel 및 image 파일을 생성합니다.

In [ ]:
import pandas as pd
import numpy as np
from PIL import Image, ImageDraw
import os

# sample sales data가 포함된 대용량 Excel 파일 생성
np.random.seed(42)
data = {
    "Date": pd.date_range("2023-01-01", periods=1000, freq="h"),
    "Product": np.random.choice(["Widget A", "Widget B", "Widget C", "Gadget X", "Gadget Y"], 1000),
    "Sales": np.random.randint(1, 1000, 1000),
    "Revenue": np.random.uniform(10.0, 5000.0, 1000),
    "Region": np.random.choice(["North", "South", "East", "West"], 1000),
    "Customer_ID": np.random.randint(1000, 9999, 1000),
}

df = pd.DataFrame(data)
df.to_excel("large_sales_data.xlsx", index=False)

# sample chart image 생성
img = Image.new("RGB", (600, 500), color="white")
draw = ImageDraw.Draw(img)

# 간단한 bar chart 그리기
products = ["Widget A", "Widget B", "Widget C", "Gadget X", "Gadget Y"]
values = [250, 180, 320, 150, 280]
colors = ["#FF6B6B", "#4ECDC4", "#45B7D1", "#96CEB4", "#FFEAA7"]

max_value = max(values)
bar_width = 120
start_x = 100

for i, (product, value, color) in enumerate(zip(products, values, colors)):
    x = start_x + i * (bar_width + 20)
    height = int((value / max_value) * 400)
    y = 500 - height

    # bar 그리기
    draw.rectangle([x, y, x + bar_width, 500], fill=color)

    # label 추가(font 없이 단순화)
    draw.text((x + 10, 510), product[:8], fill="black")
    draw.text((x + 10, y - 20), str(value), fill="black")

draw.text((300, 50), "Sales Performance by Product", fill="black")
img.save("sales_chart.png")

# 파일 크기 확인
excel_size = os.path.getsize("large_sales_data.xlsx") / (1024 * 1024)  # MB
image_size = os.path.getsize("sales_chart.png") / (1024 * 1024)  # MB

print(f"Excel file size: {excel_size:.2f} MB")
print(f"Image file size: {image_size:.2f} MB")
print(f"Total payload size: {excel_size + image_size:.2f} MB")

## Multi-Modal Agent 생성

대용량 payload의 Excel 파일과 image를 모두 처리할 수 있는 에이전트를 생성합니다.

In [ ]:
%%writefile multimodal_data_agent.py
from strands import Agent, tool
from strands.models import BedrockModel
import pandas as pd
import base64
import io
import json
from bedrock_agentcore.runtime import BedrockAgentCoreApp

app = BedrockAgentCoreApp()

# 모델 및 에이전트 초기화
model_id = "global.anthropic.claude-haiku-4-5-20251001-v1:0"
model = BedrockModel(
    model_id=model_id,
    max_tokens=16000
)

agent = Agent(
    model=model,
    system_prompt="""
    You are a data analysis assistant that can process large Excel files and images.
    When given multi-modal data, analyze both the structured data and visual content,
    then provide comprehensive insights combining both data sources.
    """
)

@app.entrypoint
def multimodal_data_processor(payload, context):
    """
    Excel 데이터와 이미지가 포함된 대용량 멀티모달 payload를 처리합니다.
    
    매개변수:
        payload: prompt, excel_data(base64), image_data(base64)를 포함한 입력
        context: Runtime context 정보
    
    반환값:
        str: 두 데이터 소스의 분석 결과
    """
    prompt = payload.get("prompt", "Analyze the provided data.")
    excel_data = payload.get("excel_data", "")
    image_data = payload.get("image_data", "")
    
    print(f"=== Large Payload Processing ===")
    print(f"Session ID: {context.session_id}")
    
    if excel_data:
        print(f"Excel data size: {len(excel_data) / 1024 / 1024:.2f} MB")
    if image_data:
        print(f"Image data size: {len(image_data) / 1024 / 1024:.2f} MB")
    print(f"Excel data {excel_data}")
    print(f"Image data {image_data}")
    print(f"=== Processing Started ===")
    # base64를 byte로 decode
    excel_bytes = base64.b64decode(excel_data)
    # base64를 byte로 decode
    image_bytes = base64.b64decode(image_data)
    
    # data context가 포함된 향상된 prompt
    enhanced_prompt = f"""{prompt}
    Please analyze both data sources and provide insights.
    """
    
    response = agent(
        [{
            "document": {
                "format": "xlsx",
                "name": "excel_data",
                "source": {
                    "bytes": excel_bytes
                }
            }
        },
        {
            "image": {
                "format": "png",
                "source": {
                    "bytes": image_bytes
                }
            }
        },
        {
            "text": enhanced_prompt
        }]
    )
    return response.message['content'][0]['text']

if __name__ == "__main__":
    app.run()

## Infrastructure 설정 및 에이전트 배포

In [ ]:
from bedrock_agentcore_starter_toolkit import Runtime
from boto3.session import Session

boto_session = Session()
region = boto_session.region_name

agentcore_runtime = Runtime()

response = agentcore_runtime.configure(
    entrypoint="multimodal_data_agent.py",
    auto_create_execution_role=True,
    auto_create_ecr=True,
    requirements_file="requirements.txt",
    region=region,
    agent_name="multimodal_data_agent",
)

launch_result = agentcore_runtime.launch()

In [ ]:
import time

status_response = agentcore_runtime.status()
status = status_response.endpoint["status"]
end_status = ["READY", "CREATE_FAILED", "DELETE_FAILED", "UPDATE_FAILED"]

while status not in end_status:
    time.sleep(10)
    status_response = agentcore_runtime.status()
    status = status_response.endpoint["status"]
    print(f"Deployment status: {status}")

print(f"Final status: {status}")

## 대용량 Multi-Modal Payload 테스트

Excel 데이터와 image가 모두 포함된 대용량 payload로 에이전트를 테스트합니다.

In [ ]:
import base64
import uuid
import json
from IPython.display import Markdown, display

# 파일을 base64로 encode
with open("large_sales_data.xlsx", "rb") as f:
    excel_base64 = base64.b64encode(f.read()).decode("utf-8")

with open("sales_chart.png", "rb") as f:
    image_base64 = base64.b64encode(f.read()).decode("utf-8")

# 대용량 payload 생성
large_payload = {
    "prompt": "Analyze the sales data from the Excel file and correlate it with the chart image. Provide insights on sales performance and trends.",
    "excel_data": excel_base64,
    "image_data": image_base64,
}

session_id = str(uuid.uuid4())
print("📊 Processing large multi-modal payload...")
print(f"📋 Session ID: {session_id}")
print(f"📄 Excel size: {len(excel_base64) / 1024 / 1024:.2f} MB")
print(f"🖼️ Image size: {len(image_base64) / 1024 / 1024:.2f} MB")
print(f"📦 Total payload: {len(json.dumps(large_payload)) / 1024 / 1024:.2f} MB\n")

# 대용량 payload로 에이전트 호출
invoke_response = agentcore_runtime.invoke(large_payload, session_id=session_id)
final_response = ""
for r in invoke_response["response"]:
    final_response += r
response_data = final_response
display(Markdown(response_data))

## 리소스 정리

In [ ]:
import boto3

# AWS 리소스 정리
agentcore_control_client = boto3.client("bedrock-agentcore-control", region_name=region)
ecr_client = boto3.client("ecr", region_name=region)

# AgentCore Runtime 삭제
runtime_delete_response = agentcore_control_client.delete_agent_runtime(agentRuntimeId=launch_result.agent_id)

# ECR repository 삭제
ecr_client.delete_repository(repositoryName=launch_result.ecr_uri.split("/")[1], force=True)

# 로컬 파일 정리
os.remove("large_sales_data.xlsx")
os.remove("sales_chart.png")

print("✅ Cleanup completed!")

# 축하합니다!

Amazon Bedrock AgentCore Runtime으로 대용량 multi-modal payload를 처리하는 방법을 성공적으로 살펴봤습니다.

## 학습한 내용:

### 대용량 Payload 처리
* **100MB 지원**: AgentCore Runtime에서 최대 100MB의 payload 처리
* **Base64 Encoding**: JSON payload를 통한 binary data의 안전한 전송
* **효율적인 처리**: 대용량 데이터 처리에 최적화된 Runtime

### Multi-Modal 기능
* **Excel 분석**: spreadsheet의 structured data 처리
* **Image 처리**: 시각적 콘텐츠와 chart 분석
* **통합 분석**: 여러 data type의 insight 연계

### 주요 이점
* **Rich Data 처리**: 복잡한 multi-format dataset 처리
* **확장 가능한 아키텍처**: 대규모 workload용으로 설계된 Runtime
* **Tool 통합**: 전문 데이터 처리를 위한 custom tool
* **Enterprise 지원**: 민감한 비즈니스 데이터의 안전한 처리

이는 AgentCore Runtime이 여러 data modality를 사용하는 enterprise 규모의 데이터 처리 작업을 지원함을 보여주며, 복잡한 business intelligence 및 데이터 분석 애플리케이션에 적합합니다.